# 02.05 OCR 与 LLM 集成

## 本节概述

<table style="text-align: left; margin-left: 0;">
<tr><td align="left"><b>前置要求</b></td><td align="left">已完成 02.02-02.04 语音部分，已开通 OCR 服务</td></tr>
<tr><td align="left"><b>本节目标</b></td><td align="left">用华为云 OCR 识别图片文字，集成 LLM 做对话与工具调用</td></tr>
<tr><td align="left"><b>本节内容</b></td><td align="left">通用文字识别 → LLM 对话 → 工具调用（Function Calling）</td></tr>
</table>

## 第一部分：文字识别（OCR）


In [ ]:
# 安装华为云 OCR SDK
# ⚠️ huaweicloudsdkocr 对 Python 3.12+ 兼容性不佳；当前 cann_py311 内核（3.11.4）满足要求
!pip install huaweicloudsdkocr -q

print("✅ OCR SDK 安装完成")


## 1. 通用文字识别

通用文字识别是最常用的 OCR 能力。流程：**读取图片 → Base64 编码 → 调用 API → 解析返回的文字**。

定义通用文字识别函数：

In [ ]:
# 💡 如果 .env 尚未创建，请先运行 02.02 的第一个 cell 创建并填入凭证
import os, base64
from dotenv import load_dotenv
load_dotenv()

from huaweicloudsdkcore.auth.credentials import BasicCredentials
from huaweicloudsdkocr.v1.region.ocr_region import OcrRegion
from huaweicloudsdkocr.v1 import OcrClient, RecognizeGeneralTextRequest, GeneralTextRequestBody

ak = os.getenv('HUAWEI_SIS_AK', '')
sk = os.getenv('HUAWEI_SIS_SK', '')
region = os.getenv('HUAWEI_SIS_REGION', 'cn-east-3')


def ocr_general_text(image_path):
    """
    通用文字识别

    参数:
        image_path: 图片文件路径
    返回:
        识别出的文字行列表
    """
    credentials = BasicCredentials(ak, sk)
    client = OcrClient.new_builder() \
        .with_credentials(credentials) \
        .with_region(OcrRegion.value_of(region)) \
        .build()

    # 读取图片并 Base64 编码（OCR API 要求）
    with open(image_path, "rb") as f:
        image_base64 = base64.b64encode(f.read()).decode("utf-8")

    request = RecognizeGeneralTextRequest()
    request.body = GeneralTextRequestBody(image=image_base64)

    response = client.recognize_general_text(request)
    return response


# 测试：先用一张带文字的测试图
# 如果没有现成图片，可以用 OpenCV 生成一张含文字的图
import cv2, numpy as np
test_img = np.full((200, 600, 3), 255, dtype=np.uint8)   # 白底
cv2.putText(test_img, "Hello Huawei OCR 2024", (50, 110),
            cv2.FONT_HERSHEY_SIMPLEX, 1.2, (0, 0, 0), 3)
cv2.imwrite("ocr_test.png", test_img)
print("✅ 测试图已生成 ocr_test.png")

# 识别
try:
    result = ocr_general_text("ocr_test.png")
    print("\n识别结果:")
    print(result)
except Exception as e:
    print(f"识别失败: {e}")
    print("💡 请检查：1) AK/SK 是否正确 2) 是否开通 OCR 服务 3) 区域是否为 cn-east-3")


### 返回结果解读

OCR 返回的 `result` 中，`words_result` 是一个列表，每个元素代表一行文字，含：
- `words`：该行文字内容
- `location`：文字在图片中的位置坐标

## 2. 拍照识别流程（需本地摄像头）

实际应用中，OCR 常与摄像头结合：拍照 → 识别。完整流程：

> ⚠️ 拍照步骤需要本地摄像头，云环境无摄像头。云环境可直接用已有的图片文件测试 OCR。

In [ ]:
# 拍照 + OCR 流程（需本地摄像头）
# 本地运行时取消注释

# import cv2, time
#
# # Step 1: 拍照
# cap = cv2.VideoCapture(0)
# time.sleep(1)   # 摄像头预热
# ret, frame = cap.read()
# cap.release()
#
# if ret:
#     cv2.imwrite("capture.jpg", frame)
#     print("✅ 拍照成功: capture.jpg")
#
#     # Step 2: OCR 识别
#     result = ocr_general_text("capture.jpg")
#     print("识别到的文字:")
#     for item in result.words_result:
#         print(f"  {item.words}")
# else:
#     print("拍照失败")

print("⚠️ 拍照步骤需本地摄像头，云环境用已有图片测试 OCR 即可")
# ===== 云环境替代方案：用代码生成测试图代替拍照 =====
import cv2, numpy as np
test_img = np.full((200, 600, 3), 255, dtype=np.uint8)
cv2.putText(test_img, 'Hello Huawei OCR 2024', (50, 110),
            cv2.FONT_HERSHEY_SIMPLEX, 1.2, (0, 0, 0), 3)
cv2.imwrite('capture.jpg', test_img)
print('✅ 已生成测试图 capture.jpg 代替拍照')
result = ocr_general_text('capture.jpg')
print('OCR 识别结果:', result)

print("本地运行时取消上方注释")


## 3. 结构化识别（身份证/表格）

华为云 OCR 还支持身份证、表格等结构化识别，返回的是**字段化的结构**（如身份证返回姓名、性别、民族、出生日期等字段），而非纯文字行。

以身份证识别为例（仅演示 API，不使用真实证件）：

In [ ]:
# 证件/文档识别（用通用文字识别 OCR，已订阅可用）
# 演示：用通用 OCR 识别虚拟身份证上的文字
# 💡 通用 OCR 能识别任意图片上的文字，包括证件、表格、文档等
from IPython.display import Image, display

# 显示测试图（虚拟身份证，假信息，纯教学用）
print("📋 证件识别测试图（虚拟身份证·样证，纯教学用，无真实隐私）：")
display(Image("./images/id_card_sample.jpg", width=500))

# 用通用 OCR 识别（ocr_general_text 在 Cell[3] 已定义）
print("\n🔍 调用华为云通用文字 OCR 识别证件内容...")
try:
    result = ocr_general_text("./images/id_card_sample.jpg")
    print("✅ OCR 识别出的文字内容：")
    print("=" * 50)
    # result 是识别结果，提取文字行打印
    if hasattr(result, 'result') and hasattr(result.result, 'words_block_list'):
        # SDK 返回的对象格式
        for i, block in enumerate(result.result.words_block_list, 1):
            words = block.words if hasattr(block, 'words') else str(block)
            print(f"  行{i}: {words}")
    elif isinstance(result, dict):
        # 字典格式
        for k, v in result.items():
            print(f"  {k}: {v}")
    else:
        print(result)
    print("=" * 50)
    print("\n💡 通用 OCR 把图片上的文字逐行识别出来了。")
    print("   生产环境若需结构化字段（姓名/性别/身份证号分别提取），可订阅专用证件 OCR API。")
except Exception as e:
    print(f"⚠️ 识别失败: {e}")
    print("💡 可能原因：1) 未开通通用文字识别 OCR 服务 2) 网络问题")
    print("   开通地址：https://console.huaweicloud.com/ocr")

---

## 本节练习

**练习 1（选择）**：OCR 的作用是什么？
- A. 把文字转成图片
- B. 从图片中提取文字
- C. 翻译图片中的文字
- D. 压缩图片

**练习 2（填空）**：调用华为云 OCR API 时，图片需要先 ______ 编码后传入请求；返回结果中，`______` 字段存放识别出的文字行列表。

**练习 3（代码）**：写一段代码，对一张图片做 OCR，**统计识别到的总字符数**（所有文字行 words 字段拼接后的长度）。

> 💡 参考答案见下方 code cell。

In [ ]:
# 查看本节练习答案
!cat ./answer/02.05_ocr_llm/answers_ocr.txt



---

## 第二部分：LLM 集成与工具调用


In [ ]:
import os
from dotenv import load_dotenv
load_dotenv()
from openai import OpenAI

# 初始化 LLM 客户端（DeepSeek 官方 API，兼容 OpenAI 格式）
# DeepSeek API Key 从 .env 读取（02.02 已创建模板）
deepseek_api_key = os.getenv('DEEPSEEK_API_KEY', 'your-api-key')
llm_client = OpenAI(
    api_key=deepseek_api_key,
    base_url="https://api.deepseek.com/v1"   # DeepSeek 官方 API 地址
)


def chat_with_llm(messages, model='deepseek-v4-flash', temperature=0.7, stream=False):
    """
    与大语言模型对话

    参数:
        messages: 对话历史 [{"role": "system", "content": "..."}, {"role": "user", "content": "..."}]
        model: 模型名称（deepseek-v4-flash 通用对话快 / deepseek-v4-pro 能力更强）
        temperature: 创造性程度（0 严谨，1 随性）
        stream: 是否流式输出
    返回:
        模型回复文本
    """
    response = llm_client.chat.completions.create(
        model=model,
        messages=messages,
        temperature=temperature,
        stream=stream,
    )
    if stream:
        # 流式输出：逐字符打印
        result = ""
        for chunk in response:
            if chunk.choices[0].delta.content:
                content = chunk.choices[0].delta.content
                result += content
                print(content, end="", flush=True)
        print()
        return result
    else:
        return response.choices[0].message.content


# ===== 测试 1：普通对话 =====
print("=" * 50)
print("测试 1：普通对话（非流式）")
print("=" * 50)
messages = [
    {"role": "system", "content": "你是一个友好的 AI 助手，用简洁的中文回答。"},
    {"role": "user", "content": "请用一句话介绍华为云。"}
]
print("👤 用户：请用一句话介绍华为云。")
print("🤖 LLM 回复：")
reply = chat_with_llm(messages)
print(reply)
print()

# ===== 测试 2：流式输出（模拟大模型实时生成）=====
print("=" * 50)
print("测试 2：流式输出（逐字打印，像 ChatGPT 那样）")
print("=" * 50)
messages2 = [
    {"role": "system", "content": "你是华为云技术专家。"},
    {"role": "user", "content": "用三句话说明昇腾 NPU 相比 GPU 的优势。"}
]
print("👤 用户：用三句话说明昇腾 NPU 相比 GPU 的优势。")
print("🤖 LLM 回复（流式）：")
reply2 = chat_with_llm(messages2, stream=True)

### messages 的角色说明

<table style="text-align: left; margin-left: 0;">
<tr style="background-color:#f0f0f0">
  <th align="left">role</th><th align="left">含义</th><th align="left">示例</th></tr>
<tr><td align="left"><code>system</code></td><td align="left">系统设定，定义 AI 的人设和行为</td><td align="left">"你是一个友好的助手"</td></tr>
<tr><td align="left"><code>user</code></td><td align="left">用户的输入</td><td align="left">"今天天气怎么样？"</td></tr>
<tr><td align="left"><code>assistant</code></td><td align="left">AI 之前的回复（多轮对话时保留）</td><td align="left">"今天晴，25度"</td></tr>
</table>

## 1. 工具调用（Function Calling）

大语言模型不仅能对话，还能**调用外部工具**——这是 AI Agent 的基础范式。

### 工作流程

```text
用户提问 → LLM 判断是否需要工具 → 生成工具调用参数 → 执行工具 → 把结果返回给 LLM → 最终回复
```

以"天气查询 Agent"为例：用户问"北京天气如何"，LLM 自己不知道实时天气，但知道有个 `get_weather` 工具可以查，于是生成调用参数 `{location: "北京"}`，我们执行工具拿到结果，再让 LLM 总结成自然语言。

### 定义工具函数

先定义两个工具：查经纬度、查温度（用免费的 Open-Meteo API）：

In [ ]:
import requests

# 中文地名 → 英文映射（Open-Meteo geocoding API 不支持中文，需先转英文）
# 这里内置常见城市，其他地名走 fallback（LLM 提取地名时已转英文）
CITY_CN_TO_EN = {
    "北京": "Beijing", "上海": "Shanghai", "广州": "Guangzhou", "深圳": "Shenzhen",
    "成都": "Chengdu", "杭州": "Hangzhou", "武汉": "Wuhan", "西安": "Xian",
    "南京": "Nanjing", "重庆": "Chongqing", "天津": "Tianjin", "苏州": "Suzhou",
    "长沙": "Changsha", "青岛": "Qingdao", "哈尔滨": "Harbin", "昆明": "Kunming",
    "大连": "Dalian", "郑州": "Zhengzhou", "济南": "Jinan", "福州": "Fuzhou",
    "厦门": "Xiamen", "合肥": "Hefei", "南昌": "Nanchang", "贵阳": "Guiyang",
    "兰州": "Lanzhou", "太原": "Taiyuan", "海口": "Haikou", "三亚": "Sanya",
    "拉萨": "Lhasa", "乌鲁木齐": "Urumqi", "呼和浩特": "Hohhot", "银川": "Yinchuan",
    "西宁": "Xining", "南宁": "Nanning", "石家庄": "Shijiazhuang", "沈阳": "Shenyang",
    "长春": "Changchun", "香港": "Hong Kong", "澳门": "Macau", "台北": "Taipei",
}


def get_location_coordinates(location):
    """通过地名获取经纬度（使用 Open-Meteo 免费地理编码 API）"""
    # 中文地名先转英文（Open-Meteo geocoding 不支持中文）
    if location in CITY_CN_TO_EN:
        location = CITY_CN_TO_EN[location]
    url = f"https://geocoding-api.open-meteo.com/v1/search?name={location}&count=1&language=zh"
    response = requests.get(url, timeout=10)
    data = response.json()
    if data.get('results'):
        result = data['results'][0]
        return result['latitude'], result['longitude'], result.get('name', location)
    return None, None, None


def get_current_temperature(latitude, longitude, unit="celsius"):
    """通过经纬度获取当前温度（Open-Meteo 免费天气 API）"""
    url = (f"https://api.open-meteo.com/v1/forecast?"
           f"latitude={latitude}&longitude={longitude}"
           f"&current=temperature_2m&timezone=auto")
    response = requests.get(url, timeout=10)
    data = response.json()
    return data.get('current', {}).get('temperature_2m', '未知')


def get_weather(location):
    """组合工具：地名 → 经纬度 → 温度（支持中文/英文地名）"""
    lat, lon, name = get_location_coordinates(location)
    if lat is None:
        return f"找不到地点: {location}"
    temp = get_current_temperature(lat, lon)
    return f"{name} 当前温度: {temp}°C (经纬度: {lat:.2f}, {lon:.2f})"


# 测试工具
print("=== 天气工具测试 ===")
print(get_weather("Beijing"))   # 英文
print(get_weather("上海"))       # 中文（走映射表）
print(get_weather("深圳"))       # 中文（走映射表）

### 让 LLM 调用工具

现在让 LLM 根据用户问题决定是否调用 `get_weather`：

In [ ]:
# 简化的工具调用流程（手动版，便于理解原理）
def weather_agent(user_query):
    """
    天气查询 Agent：根据用户问题查天气并回答

    这是一个简化版的手动工具调用，便于理解原理。
    生产环境可用 LLM 原生的 function_calling 能力自动化。
    """
    # Step 1: 让 LLM 从用户问题中提取地名
    extract_msg = [
        {"role": "system", "content": "你是地名提取器。从用户问题中提取要查询的地名，只输出地名，不要其他内容。如果无法提取就输出'NONE'。"},
        {"role": "user", "content": user_query}
    ]
    location = chat_with_llm(extract_msg, temperature=0).strip()

    if location == 'NONE' or not location:
        return "抱歉，我没听出你想查哪个地方的天气，请说明具体城市。"

    print(f"  [Agent] 识别到地名: {location}")

    # Step 2: 调用工具查天气
    weather_info = get_weather(location)
    print(f"  [Agent] 工具返回: {weather_info}")

    # Step 3: 让 LLM 用自然语言总结
    summary_msg = [
        {"role": "system", "content": "你是天气助手，用友好的中文回答用户。"},
        {"role": "user", "content": f"用户问: {user_query}\n查询到的信息: {weather_info}\n请用自然语言回答用户。"}
    ]
    reply = chat_with_llm(summary_msg, temperature=0.7)
    return reply


# 测试 Agent
print("\n=== 天气查询 Agent 测试 ===")
print("\n用户: 北京天气如何？")
print("助手:", weather_agent("北京天气如何？"))


### Agent 的价值

普通 LLM 只能基于训练数据回答（不知道实时信息）；而 Agent 通过**工具调用**，能获取实时数据、操作外部系统，能力大幅扩展：

<table style="text-align: left; margin-left: 0;">
<tr style="background-color:#f0f0f0">
  <th align="left">能力</th><th align="left">普通 LLM</th><th align="left">LLM Agent</th></tr>
<tr><td align="left">实时天气</td><td align="left">❌ 不知道</td><td align="left">✅ 调天气 API</td></tr>
<tr><td align="left">查询数据库</td><td align="left">❌ 无法访问</td><td align="left">✅ 调 SQL 工具</td></tr>
<tr><td align="left">发邮件/消息</td><td align="left">❌ 无法执行</td><td align="left">✅ 调通讯工具</td></tr>
</table>

---

## 本节练习

**练习 1（选择）**：LLM 中 `messages` 的 `system` 角色作用是？
- A. 存储用户输入
- B. 定义 AI 的人设和行为规则
- C. 存储工具调用结果
- D. 记录系统日志

**练习 2（填空）**：Function Calling（工具调用）的工作流程是：用户提问 → LLM 判断是否需要工具 → ______ → 执行工具 → ______ → 最终回复。

**练习 3（代码）**：参考天气 Agent 的写法，实现一个"翻译 Agent"——用户说中文，LLM 判断目标语言并调用翻译工具（其实就是 LLM 自己翻译），返回结果。

> 💡 参考答案见下方 code cell。

In [ ]:
# 查看本节练习答案
!cat ./answer/02.05_ocr_llm/answers_llm.txt
